# Nhóm 1 — MLP PyTorch (Google Colab)
Notebook chỉ điều hợp nền tảng; toàn bộ mô hình và vòng lặp dùng chung nằm trong `src/` và `scripts/run_group1_mlp.py`.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL03')
PROJECT_ROOT = Path('/content/BTL03')
if not (PROJECT_ROOT / 'scripts/run_group1_mlp.py').is_file():
    raise FileNotFoundError('Checkout repository vào /content/BTL03 trước khi chạy notebook.')
print('Drive artifact root:', DRIVE_ROOT)

Nếu repo đã được checkout vào `/content/BTL03`, cell sau dùng trực tiếp; nếu không, đặt bản checkout tại đường dẫn đó trước khi chạy.

In [ ]:
import os, sys, subprocess
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
!python scripts/download_data.py check

## T0201 — Gaussian MLP và sweep A/B/C
Chạy đủ sweep gốc; kết quả CSV và TensorBoard nằm ở `experiments/`. Bề mặt chuẩn và đường cong loss được lưu trong `outputs/figures/`.

In [ ]:
!python scripts/run_group1_mlp.py --task func

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('experiments/csv_logs/t0201_func_apxm_abc_sweep.csv'))
Image(filename='outputs/figures/t0201_gaussian_surface.png')

## T0202 — MNIST MLP
Dùng SGD, lr=0.01, batch=128, 30 epochs. Nếu torchvision không khả dụng, runner dùng parser IDX của repo để tải cùng bộ dữ liệu MNIST chính thức. Checkpoint/features được mirror vào Drive nếu thư mục mount phù hợp.

In [ ]:
!python scripts/run_group1_mlp.py --task mnist

In [ ]:
import json, shutil
metrics = json.load(open('outputs/features/mnist_mlp_metrics.json', encoding='utf-8'))
print({k:v for k,v in metrics.items() if k != 'history'})
for relative in [metrics['checkpoint'], metrics['features']]:
    source = PROJECT_ROOT / relative
    destination = DRIVE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
print('Đã đồng bộ checkpoint và fc2 features lên Drive.')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/tensorboard/Group1_MLP